# Partie B — Préparation des données

Objectif : Nettoyer, corriger et enrichir le dataset pour le rendre exploitable.
Chaque choix de prétraitement est justifié par une logique métier ou statistique.

In [1]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

df = pd.read_csv('../data/raw/marketing_campaign.csv', sep=';')
print(f"Dataset chargé : {df.shape[0]} lignes x {df.shape[1]} colonnes")

Dataset chargé : 2240 lignes x 29 colonnes


In [2]:
# ── 1. SUPPRESSION VARIABLES INUTILES ────────────────────
# Z_CostContact et Z_Revenue sont des constantes (valeur unique)
# ID n'apporte aucune information prédictive
df.drop(columns=['Z_CostContact', 'Z_Revenue', 'ID'], inplace=True)
print(f"Après suppression colonnes inutiles : {df.shape[1]} colonnes")

# ── 2. TRAITEMENT VALEURS MANQUANTES ─────────────────────
# Income : 24 manquants (1.07%) -> imputation par la médiane
# Justification : distribution asymétrique, médiane plus robuste que moyenne
df['Income'].fillna(df['Income'].median(), inplace=True)
print(f"Valeurs manquantes restantes : {df.isnull().sum().sum()}")

# ── 3. SUPPRESSION OUTLIERS ───────────────────────────────
# Income > 200 000 : valeur aberrante (666 666) -> erreur de saisie évidente
# Year_Birth < 1930 : âge > 94 ans -> non représentatif
avant = len(df)
df = df[df['Income'] < 200_000]
df = df[df['Year_Birth'] >= 1930]
print(f"Outliers supprimés : {avant - len(df)} lignes")
print(f"Dataset après nettoyage : {df.shape[0]} lignes")

Après suppression colonnes inutiles : 26 colonnes
Valeurs manquantes restantes : 24
Outliers supprimés : 28 lignes
Dataset après nettoyage : 2212 lignes


In [3]:
# ── 4. CREATION DE NOUVELLES VARIABLES ───────────────────

# Age du client
df['Age'] = 2024 - df['Year_Birth']

# Ancienneté client en jours
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])
date_ref = df['Dt_Customer'].max()
df['Seniority_Days'] = (date_ref - df['Dt_Customer']).dt.days

# Total dépenses toutes catégories
df['TotalSpend'] = (df['MntWines'] + df['MntFruits'] +
                    df['MntMeatProducts'] + df['MntFishProducts'] +
                    df['MntSweetProducts'] + df['MntGoldProds'])

# Total enfants au foyer
df['TotalChildren'] = df['Kidhome'] + df['Teenhome']

# Situation familiale simplifiée
# Justification : regroupement des modalités parasites (Absurd, YOLO, Alone)
couple = ['Together', 'Married']
df['Is_Couple'] = df['Marital_Status'].apply(lambda x: 1 if x in couple else 0)

# Taux d'acceptation campagnes précédentes
df['CmpAccepted_Total'] = (df['AcceptedCmp1'] + df['AcceptedCmp2'] +
                            df['AcceptedCmp3'] + df['AcceptedCmp4'] +
                            df['AcceptedCmp5'])

# Part du web dans les achats totaux
df['WebRatio'] = df['NumWebPurchases'] / (
    df['NumWebPurchases'] + df['NumCatalogPurchases'] +
    df['NumStorePurchases'] + 1)

# Dépense moyenne par achat
df['AvgSpendPerPurchase'] = df['TotalSpend'] / (
    df['NumWebPurchases'] + df['NumCatalogPurchases'] +
    df['NumStorePurchases'] + 1)

print("Variables créées :")
new_vars = ['Age', 'Seniority_Days', 'TotalSpend', 'TotalChildren',
            'Is_Couple', 'CmpAccepted_Total', 'WebRatio', 'AvgSpendPerPurchase']
for v in new_vars:
    print(f"   - {v} : {df[v].describe()[['mean','min','max']].to_dict()}")

Variables créées :
   - Age : {'mean': 55.08634719710669, 'min': 28.0, 'max': 84.0}
   - Seniority_Days : {'mean': 353.7142857142857, 'min': 0.0, 'max': 699.0}
   - TotalSpend : {'mean': 607.2680831826401, 'min': 5.0, 'max': 2525.0}
   - TotalChildren : {'mean': 0.9475587703435805, 'min': 0.0, 'max': 3.0}
   - Is_Couple : {'mean': 0.6455696202531646, 'min': 0.0, 'max': 1.0}
   - CmpAccepted_Total : {'mean': 0.298372513562387, 'min': 0.0, 'max': 4.0}
   - WebRatio : {'mean': 0.29517908050154895, 'min': 0.0, 'max': 0.9642857142857143}
   - AvgSpendPerPurchase : {'mean': 34.74949407169735, 'min': 2.0, 'max': 173.23076923076923}


In [4]:
# ── 5. ENCODAGE VARIABLES CATEGORIELLES ──────────────────
# Education : encodage ordinal (logique de niveau d'études)
education_order = {'Basic': 0, '2n Cycle': 1, 'Graduation': 2,
                   'Master': 3, 'PhD': 4}
df['Education_Enc'] = df['Education'].map(education_order)

# Suppression colonnes remplacées ou inutiles pour la modélisation
df.drop(columns=['Year_Birth', 'Dt_Customer', 'Marital_Status'], inplace=True)

print(f"Dataset final : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"\nColonnes finales :\n{list(df.columns)}")

# ── 6. SAUVEGARDE ─────────────────────────────────────────
df.to_csv('../data/processed/df_clean.csv', index=False)
print("\nDataset sauvegardé -> data/processed/df_clean.csv")

Dataset final : 2212 lignes x 32 colonnes

Colonnes finales :
['Education', 'Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Response', 'Age', 'Seniority_Days', 'TotalSpend', 'TotalChildren', 'Is_Couple', 'CmpAccepted_Total', 'WebRatio', 'AvgSpendPerPurchase', 'Education_Enc']

Dataset sauvegardé -> data/processed/df_clean.csv


In [5]:
# ── CORRECTION : imputation Income ───────────────────────
# Recharger et réappliquer proprement
df['Income'] = df['Income'].fillna(df['Income'].median())
print(f"Valeurs manquantes après correction : {df.isnull().sum().sum()}")

# Resauvegarder
df.to_csv('../data/processed/df_clean.csv', index=False)
print("Dataset mis à jour sauvegardé")

Valeurs manquantes après correction : 0
Dataset mis à jour sauvegardé
